In [1]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
sys.path.append(str(project_root))

In [2]:
import json
import ollama
from models.metric import CDE, exhaustive_CDE, EF, CDEF

## Load benchmark dataset

In [3]:
def parse_benchmark(path:str):
    
    with open(path) as f:
        dataset = json.load(f)
        dataset = dataset['Data']

    A = []
    B = []
    C = []

    for e in dataset:
        A.append(e['A'])
        B.append(e['B'])
        C.append(e['C'])

    return A,B,C

In [4]:
similarities_path = '../data/external/benchmark/test_similarities.json'
A_sim, B_sim, C_sim = parse_benchmark(similarities_path)

mix_path = '../data/external/benchmark/test_mix.json'
A_mix, B_mix, C_mix = parse_benchmark(mix_path)

types_path = '../data/external/benchmark/test_types.json'
A_type, B_type, C_type = parse_benchmark(types_path)

## Get text embeddings

In [5]:
EMBED_MODEL = 'nomic-embed-text'

def get_text_embedding_using_ollama(text:str):
    
    if text:
        response = ollama.embed(
            model=EMBED_MODEL,
            input=text,
        )

        return response["embeddings"]
    
    return None

In [6]:
def get_embeddings(entity_type:list):
    embed_type = []
    
    for entry in entity_type:
        new_entry = []
        for [entity, type] in entry:
            embed = get_text_embedding_using_ollama(entity)
            new_entry.append([embed, type])
        embed_type.append(new_entry)

    return embed_type

In [7]:
A_sim_embed = get_embeddings(A_sim)
B_sim_embed = get_embeddings(B_sim)
C_sim_embed = get_embeddings(C_sim)

A_type_embed = get_embeddings(A_type)
B_type_embed = get_embeddings(B_type)
C_type_embed = get_embeddings(C_type)

A_mix_embed = get_embeddings(A_mix)
B_mix_embed = get_embeddings(B_mix)
C_mix_embed = get_embeddings(C_mix)


## Test metric

In [8]:
def test_metric(metric_fun, A, B, C, beta = None, optimum_mini=True, verbose = False):

    n_passed = 0
    n_failed = 0

    for a, b, c in zip(A, B, C):

        if beta:
            s1 = metric_fun(a, a, beta)
            s2 = metric_fun(a, b, beta)
            s3 = metric_fun(a, c, beta)
        else:
            s1 = metric_fun(a, a)
            s2 = metric_fun(a, b)
            s3 = metric_fun(a, c)

        # if verbose: print(f'{a}\t{b}\t{c}')
        if optimum_mini:
            if s1 <= s2 <= s3:
                if verbose: print(f'TEST PASSED\t{s1}\t{s2}\t{s3}\n')
                n_passed += 1
            else:
                # if verbose: print(f'{a}\t{b}\t{c}')
                if verbose: print(f'TEST FAILED\t{s1}\t{s2}\t{s3}\n')
                n_failed += 1
        else:
            if s1 >= s2 >= s3:
                if verbose: print(f'TEST PASSED\t{s1}\t{s2}\t{s3}\n')
                n_passed += 1
            else:
                # if verbose: print(f'{a}\t{b}\t{c}')
                if verbose: print(f'TEST FAILED\t{s1}\t{s2}\t{s3}\n')
                n_failed += 1

    return n_passed, n_failed

In [9]:
p, f = test_metric(CDE, A_sim_embed, B_sim_embed, C_sim_embed, verbose=False)
print(f'SIM\tCDE\t\tTESTS PASSED: {p}\t TESTS FAILED: {f}')

p, f = test_metric(exhaustive_CDE, A_sim_embed, B_sim_embed, C_sim_embed, verbose=False)
print(f'SIM\texh_CDE\t\tTESTS PASSED: {p}\t TESTS FAILED: {f}')

p, f = test_metric(EF, A_sim_embed, B_sim_embed, C_sim_embed, verbose=False)
print(f'SIM\tEF\t\tTESTS PASSED: {p}\t TESTS FAILED: {f}')

for beta in range(0, 300, 25):
    beta = beta/100.0
    p, f = test_metric(CDEF, A_sim_embed, B_sim_embed, C_sim_embed, beta=beta, optimum_mini=False, verbose=False)
    print(f'SIM\tCDEF-{beta}\tTESTS PASSED: {p}\t TESTS FAILED: {f}')

SIM	CDE		TESTS PASSED: 112	 TESTS FAILED: 18
SIM	exh_CDE		TESTS PASSED: 66	 TESTS FAILED: 64
SIM	EF		TESTS PASSED: 81	 TESTS FAILED: 49
SIM	CDEF-0.0	TESTS PASSED: 117	 TESTS FAILED: 13
SIM	CDEF-0.25	TESTS PASSED: 119	 TESTS FAILED: 11
SIM	CDEF-0.5	TESTS PASSED: 120	 TESTS FAILED: 10
SIM	CDEF-0.75	TESTS PASSED: 117	 TESTS FAILED: 13
SIM	CDEF-1.0	TESTS PASSED: 117	 TESTS FAILED: 13
SIM	CDEF-1.25	TESTS PASSED: 117	 TESTS FAILED: 13
SIM	CDEF-1.5	TESTS PASSED: 117	 TESTS FAILED: 13
SIM	CDEF-1.75	TESTS PASSED: 117	 TESTS FAILED: 13
SIM	CDEF-2.0	TESTS PASSED: 117	 TESTS FAILED: 13
SIM	CDEF-2.25	TESTS PASSED: 117	 TESTS FAILED: 13
SIM	CDEF-2.5	TESTS PASSED: 117	 TESTS FAILED: 13
SIM	CDEF-2.75	TESTS PASSED: 117	 TESTS FAILED: 13


In [10]:
p, f = test_metric(CDE, A_type_embed, B_type_embed, C_type_embed, verbose=False)
print(f'TYPE\tCDE\t\tTESTS PASSED: {p}\t TESTS FAILED: {f}')

p, f = test_metric(exhaustive_CDE, A_type_embed, B_type_embed, C_type_embed, verbose=False)
print(f'TYPE\texh_CDE\t\tTESTS PASSED: {p}\t TESTS FAILED: {f}')

p, f = test_metric(EF, A_type_embed, B_type_embed, C_type_embed, verbose=False)
print(f'TYPE\tEF\t\tTESTS PASSED: {p}\t TESTS FAILED: {f}')

for beta in range(0, 300, 25):
    beta = beta/100.0
    p, f = test_metric(CDEF, A_type_embed, B_type_embed, C_type_embed, beta=beta, optimum_mini=False, verbose=False)
    print(f'TYPE\tCDEF-{beta}\tTESTS PASSED: {p}\t TESTS FAILED: {f}')

TYPE	CDE		TESTS PASSED: 130	 TESTS FAILED: 0
TYPE	exh_CDE		TESTS PASSED: 130	 TESTS FAILED: 0
TYPE	EF		TESTS PASSED: 81	 TESTS FAILED: 49
TYPE	CDEF-0.0	TESTS PASSED: 123	 TESTS FAILED: 7
TYPE	CDEF-0.25	TESTS PASSED: 124	 TESTS FAILED: 6
TYPE	CDEF-0.5	TESTS PASSED: 124	 TESTS FAILED: 6
TYPE	CDEF-0.75	TESTS PASSED: 124	 TESTS FAILED: 6
TYPE	CDEF-1.0	TESTS PASSED: 123	 TESTS FAILED: 7
TYPE	CDEF-1.25	TESTS PASSED: 121	 TESTS FAILED: 9
TYPE	CDEF-1.5	TESTS PASSED: 121	 TESTS FAILED: 9
TYPE	CDEF-1.75	TESTS PASSED: 121	 TESTS FAILED: 9
TYPE	CDEF-2.0	TESTS PASSED: 121	 TESTS FAILED: 9
TYPE	CDEF-2.25	TESTS PASSED: 121	 TESTS FAILED: 9
TYPE	CDEF-2.5	TESTS PASSED: 121	 TESTS FAILED: 9
TYPE	CDEF-2.75	TESTS PASSED: 121	 TESTS FAILED: 9


In [11]:
p, f = test_metric(CDE, A_mix_embed, B_mix_embed, C_mix_embed, verbose=False)
print(f'MIX\tCDE\t\tTESTS PASSED: {p}\t TESTS FAILED: {f}')

p, f = test_metric(exhaustive_CDE, A_mix_embed, B_mix_embed, C_mix_embed, verbose=False)
print(f'MIX\tCDE\t\tTESTS PASSED: {p}\t TESTS FAILED: {f}')

p, f = test_metric(EF, A_mix_embed, B_mix_embed, C_mix_embed, verbose=False)
print(f'MIX\tCDE\t\tTESTS PASSED: {p}\t TESTS FAILED: {f}')

for beta in range(0, 300, 25):
    beta = beta/100.0
    p, f = test_metric(CDEF, A_mix_embed, B_mix_embed, C_mix_embed, beta=beta, optimum_mini=False, verbose=False)
    print(f'MIX\tCDEF-{beta}\tTESTS PASSED: {p}\t TESTS FAILED: {f}')

MIX	CDE		TESTS PASSED: 108	 TESTS FAILED: 22
MIX	CDE		TESTS PASSED: 91	 TESTS FAILED: 39
MIX	CDE		TESTS PASSED: 89	 TESTS FAILED: 41
MIX	CDEF-0.0	TESTS PASSED: 126	 TESTS FAILED: 4
MIX	CDEF-0.25	TESTS PASSED: 109	 TESTS FAILED: 21
MIX	CDEF-0.5	TESTS PASSED: 112	 TESTS FAILED: 18
MIX	CDEF-0.75	TESTS PASSED: 123	 TESTS FAILED: 7
MIX	CDEF-1.0	TESTS PASSED: 126	 TESTS FAILED: 4
MIX	CDEF-1.25	TESTS PASSED: 126	 TESTS FAILED: 4
MIX	CDEF-1.5	TESTS PASSED: 117	 TESTS FAILED: 13
MIX	CDEF-1.75	TESTS PASSED: 117	 TESTS FAILED: 13
MIX	CDEF-2.0	TESTS PASSED: 117	 TESTS FAILED: 13
MIX	CDEF-2.25	TESTS PASSED: 117	 TESTS FAILED: 13
MIX	CDEF-2.5	TESTS PASSED: 117	 TESTS FAILED: 13
MIX	CDEF-2.75	TESTS PASSED: 117	 TESTS FAILED: 13
